# VSS RT-CV POT — Smoke Test Notebook

Run cells top-to-bottom on the Brev box. Each section is independently re-runnable.

**Prerequisites:** `.env` is populated, `chmod +x deepstream/init/ds-start.sh` done.

## 0. Environment

In [ ]:
import os, subprocess, json, time, pathlib

# Load HOST_IP from .env if not already in environment
env_file = pathlib.Path(".env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())

HOST_IP = os.environ.get("HOST_IP") or subprocess.check_output(
    "ip route get 1.1.1.1 | awk '{print $7; exit}'", shell=True
).decode().strip()
BACKEND = f"http://{HOST_IP}:8080"
DATA_DIR = pathlib.Path(os.environ.get("DATA_DIR", "./data"))

print(f"HOST_IP  : {HOST_IP}")
print(f"BACKEND  : {BACKEND}")
print(f"DATA_DIR : {DATA_DIR.resolve()}")

## 1. Prereq checks

In [ ]:
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

checks = [
    ("nvidia-smi",            "nvidia-smi -L"),
    ("docker",                "docker info --format '{{.ServerVersion}}'"),
    ("docker compose",        "docker compose version"),
    ("ngc CLI",               "ngc --version"),
    ("nvcr.io login",         "docker login nvcr.io --username '$oauthtoken' --password-stdin <<< $NGC_CLI_API_KEY"),
]

all_ok = True
for name, cmd in checks:
    rc, out, err = run(cmd)
    status = "OK" if rc == 0 else "FAIL"
    if rc != 0:
        all_ok = False
    snippet = out[:80] or err[:80]
    print(f"[{status:4}] {name:20} {snippet}")

print()
print("All prereqs met" if all_ok else "Fix failing checks before continuing")

## 2. Inspect vss-rt-cv image (one-time, before first `docker compose up`)

In [ ]:
TAG = os.environ.get("PERCEPTION_TAG", "3.1.0")
IMAGE = f"nvcr.io/nvidia/vss-core/vss-rt-cv:{TAG}"

# Pull if not present
rc, out, err = run(f"docker image inspect {IMAGE} --format '{{{{.Id}}}}'")
if rc != 0:
    print(f"Pulling {IMAGE} ...")
    rc, out, err = run(f"docker pull {IMAGE}")
    print(out or err)
else:
    print(f"Image present: {out[:24]}")

In [ ]:
# Check native entrypoint — critical for deciding whether ds-start.sh wraps or replaces it
rc, out, _ = run(f"docker inspect {IMAGE} --format '{{{{json .Config.Entrypoint}}}}'")
print("Entrypoint:", out)
rc2, out2, _ = run(f"docker inspect {IMAGE} --format '{{{{json .Config.Cmd}}}}'")
print("Cmd:       ", out2)

In [ ]:
# Check whether Redis proto lib is present
REDIS_LIB = "/opt/nvidia/deepstream/deepstream/lib/libnvds_redis_proto.so"
rc, out, err = run(f"docker run --rm --entrypoint ls {IMAGE} {REDIS_LIB}")
if rc == 0:
    print("[OK]  Redis proto lib found:", out)
else:
    print("[MISSING] libnvds_redis_proto.so not in image.")
    print("          See ds-start.sh comments for fallback options.")

In [ ]:
# List available proto libs so we know what broker adapters are available
rc, out, _ = run(f"docker run --rm --entrypoint find {IMAGE} /opt/nvidia/deepstream -name 'libnvds_*proto*' 2>/dev/null")
print(out or "(none found)")

## 3. Sample data

In [ ]:
videos_dir = DATA_DIR / "videos"
videos_dir.mkdir(parents=True, exist_ok=True)

existing = list(videos_dir.glob("*.mp4")) + list(videos_dir.glob("*.mkv"))
print(f"Videos already in {videos_dir}: {[f.name for f in existing] or 'none'}")

In [ ]:
# Download sample data from NGC (skip if already populated)
if not existing:
    print("Downloading sample data from NGC...")
    rc, out, err = run(
        "ngc registry resource download-version nvidia/vss-developer/dev-profile-sample-data:3.0.0"
    )
    print(out or err)

    rc2, out2, err2 = run(
        f"tar -xf dev-profile-sample-data_v3.0.0/dev-profile-sample-data.tar.gz -C {videos_dir}"
    )
    print(out2 or err2 or "Extracted.")

    run("rm -rf dev-profile-sample-data_v3.0.0")
    existing = list(videos_dir.glob("*.mp4")) + list(videos_dir.glob("*.mkv"))
    print(f"Videos now available: {[f.name for f in existing]}")
else:
    print("Sample data already present, skipping download.")

TEST_VIDEO = existing[0] if existing else None
print(f"Will use for smoke test: {TEST_VIDEO}")

## 4. Start the stack

In [ ]:
rc, out, err = run("docker compose up -d --build 2>&1")
print(out or err)

In [ ]:
# Wait for backend to be healthy
import urllib.request, urllib.error

for i in range(30):
    try:
        with urllib.request.urlopen(f"{BACKEND}/health", timeout=2) as r:
            print(f"Backend healthy: {json.loads(r.read())}")
            break
    except Exception:
        print(f"  waiting... ({i+1}/30)")
        time.sleep(2)
else:
    print("Backend did not come up — check: docker compose logs backend")

In [ ]:
# Service status overview
rc, out, _ = run("docker compose ps --format 'table {{.Name}}\t{{.Status}}\t{{.Ports}}'")
print(out)

## 5. Upload test

In [ ]:
assert TEST_VIDEO is not None, "No test video found — run section 3 first"

rc, out, err = run(
    f"curl -s -X POST {BACKEND}/api/upload "
    f"-F 'file=@{TEST_VIDEO}' | python3 -m json.tool"
)
print(out or err)

try:
    upload_result = json.loads(out)
    VIDEO_ID = upload_result["video_id"]
    RTSP_URL = upload_result["rtsp_url"]
    PLAYBACK_URL = upload_result["playback_url"]
    print(f"\nvideo_id    : {VIDEO_ID}")
    print(f"rtsp_url    : {RTSP_URL}")
    print(f"playback_url: {PLAYBACK_URL}")
except Exception as e:
    print(f"Parse error: {e}")

In [ ]:
# Verify playback endpoint serves the file
rc, out, err = run(f"curl -sI {PLAYBACK_URL}")
print(out)

## 6. NVStreamer stream check

In [ ]:
# Check NVStreamer knows about the stream
rc, out, err = run(
    f"curl -s http://{HOST_IP}:31000/api/v1/streams | python3 -m json.tool"
)
print(out or err)

## 7. Redis stream check

In [ ]:
# Wait up to 30s for detection events to start appearing
print("Waiting for detection events in mdx-raw...")
for i in range(30):
    rc, out, _ = run("docker compose exec -T redis redis-cli XLEN mdx-raw")
    count = out.strip()
    print(f"  mdx-raw length: {count}", end="\r")
    if count.isdigit() and int(count) >= 1:
        print(f"\nGot {count} events after {i+1}s")
        break
    time.sleep(1)
else:
    print("\nNo events after 30s — check: docker compose logs vss-rt-cv")

In [ ]:
# Read the last 3 raw messages — inspect actual schema
rc, out, _ = run("docker compose exec -T redis redis-cli XREVRANGE mdx-raw + - COUNT 3")
print(out)

In [ ]:
# Check total event count after ~10s of processing
time.sleep(10)
rc, out, _ = run("docker compose exec -T redis redis-cli XLEN mdx-raw")
count = int(out.strip()) if out.strip().isdigit() else 0
status = "PASS" if count >= 10 else "FAIL"
print(f"[{status}] {count} detection events (need >= 10 for POT definition of done)")

## 8. WebSocket smoke test

In [ ]:
# Requires websockets: pip install websockets
try:
    import websockets
    WS_AVAILABLE = True
except ImportError:
    WS_AVAILABLE = False
    print("websockets not installed — run: pip install websockets")

In [ ]:
if WS_AVAILABLE:
    import asyncio
    import websockets

    async def collect_events(n=5, timeout=15):
        uri = f"ws://{HOST_IP}:8080/ws/events"
        events = []
        try:
            async with websockets.connect(uri) as ws:
                print(f"Connected to {uri}")
                deadline = time.time() + timeout
                while len(events) < n and time.time() < deadline:
                    try:
                        msg = await asyncio.wait_for(ws.recv(), timeout=2)
                        events.append(json.loads(msg))
                        print(f"  event {len(events)}: {str(events[-1])[:120]}")
                    except asyncio.TimeoutError:
                        pass
        except Exception as e:
            print(f"WebSocket error: {e}")
        return events

    events = asyncio.run(collect_events(5))
    status = "PASS" if len(events) >= 5 else "FAIL"
    print(f"\n[{status}] Received {len(events)}/5 events via WebSocket")

## 9. Reset test

In [ ]:
rc, out, err = run(f"curl -s -X POST {BACKEND}/api/reset | python3 -m json.tool")
print(out or err)

time.sleep(1)
rc, out, _ = run("docker compose exec -T redis redis-cli XLEN mdx-raw")
print(f"mdx-raw length after reset: {out.strip()} (expect 0)")

## 10. POT definition-of-done summary

In [ ]:
results = {}

# Backend up
try:
    with urllib.request.urlopen(f"{BACKEND}/health", timeout=3) as r:
        results["Backend healthy"] = json.loads(r.read()).get("status") == "ok"
except:
    results["Backend healthy"] = False

# Upload works
results["Upload returns video_id"] = "VIDEO_ID" in dir() and bool(VIDEO_ID)

# Playback URL reachable
rc, out, _ = run(f"curl -so /dev/null -w '%{{http_code}}' {PLAYBACK_URL}")
results["Playback URL 200"] = out.strip() == "200"

# Detection events
rc, out, _ = run("docker compose exec -T redis redis-cli XLEN mdx-raw")
# Re-upload to get events flowing again after reset
if out.strip() == "0" and TEST_VIDEO:
    run(f"curl -s -X POST {BACKEND}/api/upload -F 'file=@{TEST_VIDEO}' > /dev/null")
    time.sleep(15)
    rc, out, _ = run("docker compose exec -T redis redis-cli XLEN mdx-raw")
event_count = int(out.strip()) if out.strip().isdigit() else 0
results[f">= 10 detection events ({event_count} seen)"] = event_count >= 10

# WebSocket
results["WebSocket delivers events"] = WS_AVAILABLE and len(events) >= 1 if WS_AVAILABLE else "skipped (no websockets lib)"

print("=" * 50)
print("POT DEFINITION OF DONE")
print("=" * 50)
all_pass = True
for label, result in results.items():
    if result == "skipped (no websockets lib)":
        icon = "--"
    elif result:
        icon = "PASS"
    else:
        icon = "FAIL"
        all_pass = False
    print(f"  [{icon:4}] {label}")
print("=" * 50)
print("POT COMPLETE" if all_pass else "NOT DONE — see failing checks above")

## 11. Useful debug commands

In [ ]:
# Tail vss-rt-cv logs (last 50 lines)
rc, out, _ = run("docker compose logs --tail=50 vss-rt-cv")
print(out)

In [ ]:
# Tail backend logs
rc, out, _ = run("docker compose logs --tail=30 backend")
print(out)

In [ ]:
# Tail SDR logs
rc, out, _ = run("docker compose logs --tail=30 sdr")
print(out)

In [ ]:
# Stream 10 live events directly from Redis (no WebSocket)
print("Streaming 10 events from mdx-raw (ctrl-c to stop early)...\n")
seen = set()
last_id = "0"
collected = 0
deadline = time.time() + 30

while collected < 10 and time.time() < deadline:
    rc, out, _ = run(f"docker compose exec -T redis redis-cli XREAD COUNT 5 STREAMS mdx-raw {last_id}")
    if out and out != "(empty array)":
        for line in out.splitlines():
            line = line.strip()
            if line and line not in seen:
                seen.add(line)
                print(line[:120])
                collected += 1
        # Advance cursor (crude — just use xrevrange for inspection)
        last_id = "$"
    time.sleep(1)

print(f"\n{collected} lines printed")